In [ ]:
# === Mount Google Drive ===
from google.colab import drive
drive.mount('/content/drive')

# === Install necessary packages (only run once) ===
!pip install -q mne plotly colorama pyphen

# === Imports ===
import os
import gc
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import pyphen
import mne
import plotly.graph_objects as go
from IPython.display import display
from colorama import Fore, Style, init

# === Colorama init (for colored terminal printouts) ===
init(autoreset=True)


# Hypotheses: Effect of Distal Speech Rate on Function Word Perception

### Null Hypothesis (H₀)

The probability of perceiving a function word is equal under slow and fast distal speech rate conditions.

$$
H_0: P(\text{Perceived} \mid \text{Slow}) = P(\text{Perceived} \mid \text{Fast})
$$

---

### Alternative Hypothesis (H₁)

The probability of perceiving a function word is greater under fast distal speech rate conditions than under slow speech rate conditions.

$$
H_1: P(\text{Perceived} \mid \text{Fast}) > P(\text{Perceived} \mid \text{Slow})
$$


# Table of Contents
- Hypotheses
- Data Preprocessing
- Feature Extraction
- Modeling & Classification
- Results & Interpretation



# 1. Data Preprocessing

## Behavioral Data Preprocessing




In [ ]:
# === 1. Parse Sentences_List1.txt ===
def parse_sentences_from_txt(txt_file):
    with open(txt_file, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    data = []
    for line in lines:
        line = line.strip()
        if line and line[0].isdigit() and '|' in line:
            try:
                idx, rest = line.split('.', 1)
                words = rest.strip().split()

                word1_raw = None
                function_word = None
                word2 = None

                for i, word in enumerate(words):
                    if '|' in word:
                        word1_raw = word
                        if i + 1 < len(words):
                            function_word = words[i + 1]
                        if i + 2 < len(words):
                            word2 = words[i + 2].replace('|', '')
                        break

                if word1_raw and function_word:
                    word1_clean = word1_raw.replace('|', '').lower()
                    function_word = function_word.lower()
                    full_sentence = ' '.join(w.replace('|', '') for w in words)

                    data.append({
                        'Index': int(idx),
                        'Word1': word1_clean,
                        'FunctionWord': function_word,
                        'Word2': word2.lower() if word2 else None,
                        'Full_Sentence': full_sentence.lower()
                    })

            except Exception as e:
                print(f"⚠️ Failed to parse line: {line}\n{e}")
                continue

    return pd.DataFrame(data)

# === 2. Check participant response logic ===
def check_sequence_after_word1(row, sentence_df):
    stim_text = str(row.get("StimTest.RESP", "")).lower().strip()
    stim_words = stim_text.split()

    idx = row.get("Index")
    match = sentence_df[sentence_df["Index"] == idx]
    if match.empty:
        return None

    word1 = match.iloc[0]["Word1"]
    func_word = match.iloc[0]["FunctionWord"]
    word2 = match.iloc[0]["Word2"]

    try:
        i = stim_words.index(word1)
    except ValueError:
        return None  # word1 not present → skip

    # case 1: Function word appears after word1
    if i + 1 < len(stim_words) and stim_words[i + 1] == func_word:
        return 0
    # case 2: word2 appears immediately after word1
    if i + 1 < len(stim_words) and stim_words[i + 1] == word2:
        return 1
    return None

# === 3. Try to read CSVs ===
def smart_read_csv(filepath):
    for enc in ['utf-8', 'utf-16', 'windows-1252']:
        for delim in [',', '\t', ';']:
            try:
                df = pd.read_csv(filepath, delimiter=delim, encoding=enc, engine='python', on_bad_lines='skip')
                if df.shape[1] >= 2:
                    return df
            except Exception:
                continue
    return None

# === 4. Main logic for printing ===
def print_evaluation_of_participants(root_dir, sentence_file):
    sentence_df = parse_sentences_from_txt(sentence_file)

    for participant_num in range(2, 28):  # 2 to 27 inclusive
        folder_path = os.path.join(root_dir, str(participant_num))
        if not os.path.isdir(folder_path):
            print(f"❌ Folder not found: {folder_path}")
            continue

        print(f"\n👤 Processing Participant {participant_num}")

        for fname in os.listdir(folder_path):
            if fname.lower().startswith("speechrate") and fname.lower().endswith(".csv"):
                input_path = os.path.join(folder_path, fname)

                df = smart_read_csv(input_path)
                if df is None:
                    print(f"❌ Could not read: {fname}")
                    continue

                df.rename(columns={"WavFiler": "Index", "CellT": "Condition"}, inplace=True)

                if not all(c in df.columns for c in ["Index", "Condition", "StimTest.RESP"]):
                    print(f"⚠️ Unexpected columns in {fname}")
                    continue

                df["Index"] = df["Index"].astype("Int64", errors="ignore")
                df = pd.merge(df, sentence_df, on="Index", how="left")

                df["Correct_Sequence (0/1)"] = df.apply(lambda row: check_sequence_after_word1(row, sentence_df), axis=1)
                df["Correct_Sequence (0/1)"] = df["Correct_Sequence (0/1)"].astype("Int64")
                df["Include_For_Stats"] = df["Correct_Sequence (0/1)"].isin([0, 1])

                # Print selected columns
                print(df[["Index", "Condition", "StimTest.RESP", "Correct_Sequence (0/1)", "Include_For_Stats"]].head(10))
                break

# === 5. Run ===
if __name__ == "__main__":
    root_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Behavioral"
    sentence_file = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/Sentences_List1.txt"
    print_evaluation_of_participants(root_dir, sentence_file)






| Value  | Meaning                                                         |
| ------ | --------------------------------------------------------------- |
| `0`    | **Correct** — The function word followed the expected `Word1` |
| `1`    | **Incorrect** — The function word **did not** follow `Word1`  |
| `<NA>` | **Not applicable** — One of the following:                   |
|        | - The response was too short                                    |
                    |
|        | - The sentence was malformed or incomplete                      |


Remove <NA> values and save the cleaned, annotated CSV file.










In [ ]:
import os
import pandas as pd
import traceback

def annotate_and_save_cleaned_participants(root_dir, sentence_file):
    print("🚀 Starting annotation process...")

    sentence_df = parse_sentences_from_txt(sentence_file)
    print(f"📘 Parsed {len(sentence_df)} sentences from sentence file.")

    for participant_num in range(2, 28):
        participant_folder = str(participant_num)
        folder_path = os.path.join(root_dir, participant_folder)

        print(f"\n📁 Checking folder: {folder_path}")

        if not os.path.isdir(folder_path):
            print(f"❌ Folder not found for participant {participant_num}")
            continue

        print(f"👤 Processing Participant {participant_num}")

        for fname in os.listdir(folder_path):
            print(f"🔍 Found file: {fname}")

            if fname.lower().startswith("speechrate") and fname.lower().endswith(".csv"):
                input_path = os.path.join(folder_path, fname)
                output_path = input_path.replace(".csv", "_cleaned_annotated.csv")

                print(f"📄 Processing file: {input_path}")

                try:
                    df = smart_read_csv(input_path)
                    if df is None or df.shape[1] < 2:
                        print(f"❌ Could not read valid table from: {fname}")
                        continue

                    df.rename(columns={"WavFiler": "Index", "CellT": "Condition"}, inplace=True)

                    expected_cols = ["Index", "Condition", "StimTest.RESP"]
                    if not all(col in df.columns for col in expected_cols):
                        print(f"⚠️ Unexpected columns in {fname}.")
                        print(f"   Found: {list(df.columns)}")
                        continue

                    df["Index"] = df["Index"].astype("Int64", errors="ignore")
                    df = pd.merge(df, sentence_df, on="Index", how="left")

                    df["Correct_Sequence (0/1)"] = df.apply(
                        lambda row: check_sequence_after_word1(row, sentence_df), axis=1)
                    df["Correct_Sequence (0/1)"] = df["Correct_Sequence (0/1)"].astype("Int64")
                    df["Include_For_Stats"] = df["Correct_Sequence (0/1)"].isin([0, 1])

                    df_cleaned = df[df["Include_For_Stats"]].copy()

                    if df_cleaned.empty:
                        print("⚠️ No usable rows after filtering. Skipping save.")
                    else:
                        df_cleaned.to_csv(output_path, index=False)
                        print(f"💾 Cleaned and saved to: {output_path}")
                        print(df_cleaned[["Index", "Condition", "StimTest.RESP", "Correct_Sequence (0/1)", "Include_For_Stats"]].head(10))

                except Exception as e:
                    print(f"❌ Error processing participant {participant_num}: {e}")
                    traceback.print_exc()

                break  # only process the first matching file

if __name__ == "__main__":
    print("✅ Script entry reached.")
    root_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Behavioral"
    sentence_file = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/Sentences_List1.txt"
    annotate_and_save_cleaned_participants(root_dir, sentence_file)



All Participants Summary:

In [ ]:

def combine_cleaned_csvs_to_summary(root_dir, output_file="All_Participants_Summary.csv"):
    all_rows = []

    for participant_num in range(2, 28):
        folder_path = os.path.join(root_dir, str(participant_num))
        if not os.path.isdir(folder_path):
            continue

        for fname in os.listdir(folder_path):
            if fname.endswith("_cleaned_annotated.csv"):
                csv_path = os.path.join(folder_path, fname)
                try:
                    df = pd.read_csv(csv_path)
                    df["Participant"] = participant_num  # Tag participant ID
                    all_rows.append(df)
                except Exception as e:
                    print(f"❌ Could not read {fname}: {e}")

    if all_rows:
        merged_df = pd.concat(all_rows, ignore_index=True)
        output_path = os.path.join(root_dir, output_file)

        # 💾 Save as CSV
        merged_df.to_csv(output_path, index=False)
        print(f"\n✅ Combined data saved to: {output_path}\n")

        # 🖨️ Show preview
        print(merged_df.head(10))
    else:
        print("⚠️ No cleaned annotated files found to combine.")

# ▶️ Run this block
if __name__ == "__main__":
    root_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Behavioral"
    combine_cleaned_csvs_to_summary(root_dir)


## Load Onset Times



In [ ]:
csv_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/SR_OnsetTimes_Final.csv"
df = pd.read_csv(csv_path)
print(df.head())



### Check If the Onset Times are in ms or seconds

In [ ]:


# === Set paths ===
audio_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli"

# === Check a sample audio file ===
sample_file = "SR129.wav"
audio_path = os.path.join(audio_dir, sample_file)

# === Load audio and print duration ===
y, sr = librosa.load(audio_path, sr=16000)  # 16kHz
duration = len(y) / sr
print(f"\n🔊 '{sample_file}' duration: {duration:.2f} seconds")

# === Check if onset values are in ms or sec ===
onset_sample_ms = df.loc[0, "critical"]  # or any other column
onset_sample_sec = onset_sample_ms / 1000
print(f"Sample 'critical' onset (converted): {onset_sample_sec:.3f} sec")

# === Conclusion ===
if onset_sample_sec < duration:
    print("✅ Onset values are in milliseconds.")
else:
    print("⚠️ Onset values might already be in seconds — please double check.")



#2. Feature Engineering

### Feature Preprocessing

###Split the regions for each sentence

In [ ]:

# === Load and parse Sentences_List1.txt ===
def parse_sentence_components(txt_path):
    with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    structured_data = []

    for line in lines:
        line = line.strip()
        if line and line[0].isdigit() and '|' in line:
            try:
                index_part, sentence = line.split('.', 1)
                index = int(index_part.strip())

                # Split the sentence into chunks using |
                chunks = sentence.strip().split('|')

                # Safety check
                if len(chunks) < 4:
                    print(f"⚠️ Skipping line {index}: not enough | segments.")
                    continue

                # Extract components
                preceding_context = chunks[0].strip()
                target_raw = chunks[1].strip()
                following_context = chunks[2].strip()
                extra_context = chunks[3].strip()

                # Target region is structured: Word1 + Function Word + start of Word2
                target_tokens = target_raw.split()
                if len(target_tokens) < 3:
                    print(f"⚠️ Skipping line {index}: target region has < 3 tokens.")
                    continue

                word1 = target_tokens[0]
                function_word = target_tokens[1]
                word2_start = target_tokens[2]

                target_region = f"{word1} {function_word} {word2_start}"

                structured_data.append({
                    "Index": index,
                    "Preceding_Context": preceding_context,
                    "Target_Region": target_region,
                    "Following_Context": following_context,
                    "Extra_Context": extra_context
                })

            except Exception as e:
                print(f"❌ Error parsing line: {line}\n{e}")
                continue

    return pd.DataFrame(structured_data)

# === Path to your sentence file ===
sentence_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/Sentences_List1.txt"

# === Run the parser ===
df_sentences = parse_sentence_components(sentence_path)

# === Save parsed output ===
csv_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_List1_parsed.csv"
df_sentences.to_csv(csv_path, index=False)

# Optional: save as Pickle (faster for later use in Python)
pkl_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_List1_parsed.pkl"
df_sentences.to_pickle(pkl_path)

# === Display the first few rows ===
from IPython.display import display
display(df_sentences.head(10))






### Counting syllables in the Preceding_Context & Computing the duration of the preceding context

In [ ]:

# === File paths ===
parsed_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_List1_parsed.csv"
onset_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/SR_OnsetTimes_Final.csv"

# === Load parsed sentences ===
df_sentences = pd.read_csv(parsed_path)

# === Load onset times and rename columns for consistency ===
df_onsets = pd.read_csv(onset_path)
df_onsets.rename(columns={
    "sentence#": "Index",
    "critical": "Target_Onset_ms"
}, inplace=True)

# === Merge on Index ===
df = pd.merge(df_sentences, df_onsets, on="Index")

# === Convert onset time to seconds ===
df["Preceding_Duration_sec"] = df["Target_Onset_ms"] / 1000.0

# === Count syllables in Preceding_Context ===
dic = pyphen.Pyphen(lang='en')

def count_syllables(text):
    return sum(len(dic.inserted(word).split('-')) for word in text.split())

df["Preceding_Syllables"] = df["Preceding_Context"].apply(count_syllables)

# === Compute Distal Speech Rate ===
df["SpeechRate_Preceding"] = df["Preceding_Syllables"] / df["Preceding_Duration_sec"]

# === Display result ===
display(df[["Index", "Preceding_Context", "Preceding_Syllables", "Preceding_Duration_sec", "SpeechRate_Preceding"]].head(10))

# === Save final DataFrame to CSV ===
output_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_with_DistalSpeechRate.csv"
df.to_csv(output_path, index=False)
print(f"Data saved to: {output_path}")



## Exploration of Different Features

| Feature                  | Can You Extract It Now?  | How? / Notes                           |
| ------------------------ | ------------------------ | -------------------------------------- |
| **Duration**             | 🔜 Yes                   | With alignment or segment extraction   |
| **Pitch (F0)**           | ✅ Yes                    | Use `librosa.pyin()` or Praat          |
| **Intensity**            | ✅ Yes                    | Use RMS energy (`librosa.feature.rms`) |
| **Formants (F1, F2)**    | 🔜 Yes (advanced)        | Use `parselmouth` (Praat in Python)    |
| **MFCCs**                | ✅ Yes                    | `librosa.feature.mfcc`                 |
| **Spectral Centroid**    | ✅ Yes                    | `librosa.feature.spectral_centroid`    |
| **Zero-Crossing Rate**   | ✅ Yes                    | `librosa.feature.zero_crossing_rate`   |
| **Speech Rate (Distal)** | ✅ Already done           | From syllables and duration            |



In [ ]:

# === Imports ===
import os
!pip install praat-parselmouth

import librosa
import parselmouth
import numpy as np
import pandas as pd
from parselmouth.praat import call
from tqdm import tqdm

# === Define paths ===
audio_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli"
output_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Extracted_Features"
os.makedirs(output_dir, exist_ok=True)

# === Feature extraction function ===
def extract_features(wav_path):
    snd = parselmouth.Sound(wav_path)
    y, sr = librosa.load(wav_path, sr=None)

    duration = snd.get_total_duration()

    # Intensity
    intensity = snd.to_intensity()
    mean_intensity = call(intensity, "Get mean", 0, 0, "energy")

    # Pitch (F0)
    pitch = snd.to_pitch()
    mean_pitch = call(pitch, "Get mean", 0, 0, "Hertz")

    # RMS energy
    rms = librosa.feature.rms(y=y).mean()

    # Zero-crossing rate
    zcr = librosa.feature.zero_crossing_rate(y).mean()

    # Spectral centroid
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr).mean()

    # MFCCs
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_means = np.mean(mfccs, axis=1)

    # Create time-series DataFrame for MFCCs
    ts_df = pd.DataFrame(mfccs.T, columns=[f"MFCC_{i+1}" for i in range(mfccs.shape[0])])

    # Summary dict
    summary = {
        "file": os.path.basename(wav_path),
        "F0_Mean": mean_pitch,
        "RMS_Mean": rms,
        "SpectralCentroid_Mean": centroid,
        "ZCR_Mean": zcr
    }
    for i, mfcc_val in enumerate(mfcc_means):
        summary[f"MFCC_{i+1}_Mean"] = mfcc_val

    return ts_df, summary

# === Process WAV files ===
summary_list = []
wav_files = [f for f in os.listdir(audio_dir) if f.endswith(".wav") and "_P" not in f]

for wav_file in tqdm(wav_files, desc="Extracting features"):
    wav_path = os.path.join(audio_dir, wav_file)
    try:
        ts_df, summary = extract_features(wav_path)

        # Save time-series MFCCs
        ts_out_path = os.path.join(output_dir, f"{os.path.splitext(wav_file)[0]}_temporal.csv")
        ts_df.to_csv(ts_out_path, index=False)

        # Add to summary
        summary_list.append(summary)

        print(f"✅ Processed: {wav_file}")
    except Exception as e:
        print(f"❌ Error processing {wav_file}: {e}")

# === Save summary features ===
summary_df = pd.DataFrame(summary_list)
summary_df_path = os.path.join(output_dir, "summary_acoustic_features.csv")
summary_df.to_csv(summary_df_path, index=False)

# === Preview summary ===
print("\n📊 Summary of Extracted Acoustic Features:")
print(summary_df.head())
print(f"\n🧮 Final summary_df shape: {summary_df.shape}")



featues - the peak rate for the envelope
the time of the first envelope peak in the critical period
and the amplitude of that first peak

the location of the peak rate
peak is the derivative of the envelope

try to predicing if its extended or normal
perceived and nor perceived

## Checking Dimentions

Checking X — Acoustic Feature Matrix

In [ ]:
import pandas as pd
import numpy as np

# === Load the acoustic features ===
summary_df = pd.read_csv("/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Extracted_Acoustic_Features/summary_acoustic_features.csv")

# If 'Index' is not present, extract it from filename like 'SR48_N.wav'
if 'Index' not in summary_df.columns:
    summary_df['Index'] = summary_df['Filename'].str.extract(r'SR(\d+)', expand=False).astype(int)

# === Load and combine behavioral trial-level data ===
root_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Behavioral"
all_trials = []

for subject_id in range(2, 28):  # ✅ Changed: Subjects 1 to 27 inclusive
    file_path = f"{root_dir}/{subject_id}/SpeechRate{subject_id}_cleaned_annotated.csv"
    try:
        df = pd.read_csv(file_path)
        if 'Correct_Sequence (0/1)' in df.columns and 'Index' in df.columns:
            df = df.dropna(subset=['Correct_Sequence (0/1)'])
            df['Subject'] = subject_id
            all_trials.append(df)
    except FileNotFoundError:
        print(f"❌ File not found for Subject {subject_id}")

# === Combine all trials into one DataFrame ===
trial_df = pd.concat(all_trials, ignore_index=True)

# === Merge trial-level data with sentence-level acoustic features ===
merged_df = pd.merge(trial_df, summary_df, on='Index')

# === Simulate ERP features === USE ORIGINAL ERP --- LOAD ---
def simulate_erp(condition):
    if isinstance(condition, str) and condition.upper() in ["N", "FAST"]:
        return np.random.normal(-3, 0.5), np.random.normal(-2.5, 0.5)
    else:
        return np.random.normal(-0.5, 0.2), np.random.normal(-0.2, 0.2)

merged_df[["ERP_N1", "ERP_N280"]] = merged_df["Condition"].apply(lambda cond: pd.Series(simulate_erp(cond)))

# === Split data by Condition ===
df_slow = merged_df[merged_df['Condition'] == 'E'].copy()
df_fast = merged_df[merged_df['Condition'] == 'N'].copy()

# === Extract features and labels ===
feature_columns = [col for col in merged_df.columns if any(f in col for f in ['MFCC', 'F0', 'ZCR', 'RMS', 'Spectral'])] + ['ERP_N1', 'ERP_N280']

X = merged_df[feature_columns].values
y = merged_df['Correct_Sequence (0/1)'].astype(int).values

# === Check dataset shapes ===
print("✅ X shape (design matrix):", X.shape)
print("✅ y shape (response vector):", y.shape)
print("📊 Shape of full dataset:", merged_df.shape)
print("🐢 Slow condition (E):", df_slow.shape)
print("⚡ Fast condition (N):", df_fast.shape)





In [ ]:
# Show a few relevant columns from the merged dataset
print(merged_df[['Subject', 'Condition', 'Index', 'Correct_Sequence (0/1)',
                 'F0_Mean', 'RMS_Mean', 'SpectralCentroid_Mean', 'ZCR_Mean',
                 'ERP_N1', 'ERP_N280']].head(10))

#0 = correct response
#1 = incorrect response


#3. Inference: Behavioral context + Acoustic features + Simulated ERP

Feature used:

| Feature Type           | Count  |
| ---------------------- | ------ |
| MFCCs                  | 13     |
| F0, RMS, ZCR, Spectral | 4      |
| Simulated ERP          | 2      |
| **Total**              | **19** |


## Classification - Logicstic Regression (19 features)



In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# === Optional: 5-Fold Cross-Validation before splitting ===
log_clf = LogisticRegression(max_iter=1000)
log_scores = cross_val_score(log_clf, X, y, cv=5)

print("📊 Logistic Regression Cross-Validation Scores (5 folds):", log_scores)
print("📈 Mean Accuracy (CV):", log_scores.mean())

# === Split data into train/test sets ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# === Train logistic regression model ===
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# === Predict and evaluate ===
y_pred = clf.predict(X_test)

print("\n🧪 Classification Report (Test Split):")
print(classification_report(y_test, y_pred))

print("📉 Confusion Matrix (Test Split):")
cm = confusion_matrix(y_test, y_pred)
print(cm)

print(f"✅ Accuracy (Test Split): {accuracy_score(y_test, y_pred):.2f}")

# === Plot labeled confusion matrix ===
labels = np.array([
    ["True Positive\n(Heard & Predicted Heard)", "False Negative\n(Heard but Predicted Missed)"],
    ["False Positive\n(Missed but Predicted Heard)", "True Negative\n(Missed & Predicted Missed)"]
])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=labels, fmt='', cmap="Blues", cbar=False,
            xticklabels=["Predicted: Heard", "Predicted: Missed"],
            yticklabels=["Actual: Heard", "Actual: Missed"])

plt.title("Confusion Matrix – Logistic Regression (Test Set)")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.show()


##Classification - XGBoost (19 features)


In [ ]:
# 📦 Install XGBoost if not already installed
!pip install xgboost
# 📦 Install XGBoost if not already installed
!pip install xgboost

# === Imports ===
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# === Optional: 5-Fold Cross-Validation before splitting ===
xgb_clf_cv = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_scores = cross_val_score(xgb_clf_cv, X, y, cv=5)

print("📊 XGBoost Cross-Validation Scores (5 folds):", xgb_scores)
print("📈 Mean Accuracy (CV):", xgb_scores.mean())

# === Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# === Train final XGBoost model ===
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_clf.fit(X_train, y_train)

# === Predict and evaluate ===
y_pred = xgb_clf.predict(X_test)

print("\n🧪 Classification Report (Test Split):")
print(classification_report(y_test, y_pred))

# === Confusion matrix ===
cm = confusion_matrix(y_test, y_pred)
print("📉 Confusion Matrix (Test Split):")
print(cm)
print(f"✅ Accuracy (Test Split): {accuracy_score(y_test, y_pred):.2f}")

# === Custom-labeled confusion matrix ===
labels = np.array([
    ["True Positive\n(Heard & Predicted Heard)", "False Negative\n(Heard but Predicted Missed)"],
    ["False Positive\n(Missed but Predicted Heard)", "True Negative\n(Missed & Predicted Missed)"]
])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=labels, fmt='', cmap="Greens", cbar=False,
            xticklabels=["Predicted: Heard", "Predicted: Missed"],
            yticklabels=["Actual: Heard", "Actual: Missed"])

plt.title("Confusion Matrix – XGBoost Classifier (Test Set)")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.show()


In [ ]:
print(merged_df['Condition'].value_counts())
print(merged_df.groupby('Condition')['Correct_Sequence (0/1)'].mean())


In [ ]:
need to correct the labels